# The 4 commands to run CESM on Derecho

```bash
# go into the scripts directory of the CESM code
cd ~/code/release-cesm2.1.5/cime/scripts

# (1) create a new case in the "cases" directory in your home directory
./create_newcase --case ~/cases/case01 --compset B1850 --res f19_g17

# go into the case you just created
cd ~/cases/case01/

# (2) setup your case
./case.setup

# (3) build the executable
qcmd -- ./case.build

# (4) submit your run
./case.submit
```


## What happens at each step

| Step | Command | What it does |
|---|---|---|
| 1 | `create_newcase` | Creates the **case directory** and the **build/run directory**. |
| 2 | `case.setup` | Sets up the case scripts. |
| 3 | `case.build` | Compiles the code. |
| 4 | `case.submit` | Runs the model (submits the job to the batch queue). |

After the run completes, history files are moved from the run directory to the archive
directory.


## How do I know it is running?

To check whether your job is running, use:
```bash
qstat -u $USER
```
```text
                                                            Req'd  Req'd   Elap
Job ID          USERname Queue    Jobname    SessID NDS TSK Memory Time  S Time
--------------- -------- -------- ---------- ------ --- --- ------ ----- - -----
6315297.desche* hannay   cpu      run.case01    --    6 768  1410g 12:00 Q   --
6315298.desche* hannay   cpu      st_archiv*    --    1   1  235gb 00:20 H   --
```

- Status `Q`: the job is queued, waiting to run.
- Status `R`: the job is running.
- No longer listed: the job is completed — or it crashed. Check the log files (see the
  [Output](../06.Output/output_overview.ipynb) chapter) to find out which!


## What does success actually look like?

"No longer listed in `qstat`" isn't proof your run succeeded — it also disappears from the
queue if it crashed. Two quick, definitive checks:

**1. `CASE/CaseStatus`** — a running log of every step, with a pass/fail line for each:
```bash
tail -20 ~/cases/case01/CaseStatus
```
```text
2026-09-02 14:32:07: case.submit starting
---------------------------------------------------
2026-09-02 14:32:09: case.run starting
---------------------------------------------------
2026-09-02 14:44:51: case.run success
---------------------------------------------------
2026-09-02 14:45:02: st_archive starting
---------------------------------------------------
2026-09-02 14:45:18: st_archive success
```
`case.run success` (not `case.run error`) is your first confirmation the model actually ran to completion.

**2. The tail of the main log file** in `$RUNDIR` (or `CASE/logs` once archived) — look for the line:
```bash
zcat ~/cases/case01/logs/cesm.log.*.gz | tail -5
```
```text
 (seq_mct_drv): =============== SUCCESSFUL TERMINATION OF CPL7-cesm ===============
 (seq_mct_drv): =============== at YMD,TOD =   00010106       0 ===============
```
`SUCCESSFUL TERMINATION` is the model itself confirming it finished cleanly. If instead the
log ends mid-timestep, with a Fortran traceback, or an `ERROR:`/`abort` message, the run
crashed — see the [Troubleshooting](../09.Troubleshooting/log_files_and_crashes.ipynb) chapter.

As a third sanity check, once `st_archive success` appears, you should see new files under
your archive directory (e.g., `.../archive/case01/rest/`).


## A view of the CESM directories after the run completes

![Overview of CESM directories after the run completes](../../images/cesm_directories_workflow.png)